Universidade Federal de Santa Catarina<br>
Departamento de Engenharia Elétrica e Eletrônica<br>
EEL7514 / EEL7513 / EEL410250 - Aprendizado de Máquina
$\newcommand{\bX}{\mathbf{X}}$
$\newcommand{\bw}{\mathbf{w}}$
$\newcommand{\by}{\mathbf{y}}$
$\newcommand{\bx}{\mathbf{x}}$
$\newcommand{\bU}{\mathbf{U}}$
$\newcommand{\bu}{\mathbf{u}}$
$\newcommand{\bT}{\mathbf{T}}$
$\newcommand{\RR}{\mathbb{R}}$
$\newcommand{\calS}{\mathcal{S}}$


# Exercício 8: Redes Convolucionais

Neste exercício você irá utilizar redes convolucionais para reconhecimento de imagens. Além de treinar uma rede a partir do zero, você irá investigar a técnica de ajuste fino (*fine tuning*) a partir de uma rede pré-treinada (*transfer learning*).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import sys
import os
import zipfile
import tarfile
import pickle

print('Python version:', sys.version.split(' ')[0])

import tensorflow as tf
print('TensorFlow version:', tf.__version__)

print("Dispositivos disponíveis:")
print(tf.config.list_physical_devices())

In [ ]:
def plots(history):
  plt.figure(figsize=(14,4))
  plt.subplot(1,2,1)
  plt.plot(history.history['loss'], '.-', label='Train loss')
  if 'val_loss' in history.history.keys():
    plt.plot(history.history['val_loss'], '.-', label='Val loss')
  plt.xlabel('Epochs');
  plt.legend();
  plt.grid();
  plt.subplot(1,2,2)
  plt.plot(history.history['accuracy'], '.-', label='Train accuracy')
  plt.xlabel('Epochs');
  if 'val_accuracy' in history.history.keys():
    plt.plot(history.history['val_accuracy'], '.-', label='Val accuracy')
  plt.legend();
  plt.grid();

# 1. Treinando a partir do zero

## MNIST

1. Assim como no exercício anterior, carregue o conjunto MNIST e separe as últimas 5000 imagens como conjunto de validação. No entanto, desta vez não realize qualquer pré-processamento nas imagens (como escalonamento); isto será feito [internamente no modelo](https://keras.io/guides/preprocessing_layers/#preprocessing-data-before-the-model-or-inside-the-model) depois.

In [ ]:
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)
print("y_test :", y_test.shape, y_test.dtype)

In [ ]:
# Separação: 55.000 imagens para treinamento e 5.000 para validação.
x_val = x_train[55000:]
y_val = y_train[55000:]

x_train = x_train[:55000]
y_train = y_train[:55000]

print("Train shapes:", x_train.shape, y_train.shape)
print("Val shapes  :", x_val.shape, y_val.shape)
print("Test shapes :", x_test.shape, y_test.shape)

2. Usando o Keras, construa uma rede neural com pelo menos uma camada convolucional (`tf.keras.layers.Conv2D`) e confirme que não há nenhum erro de definição. Organize seu código em uma função de criação do modelo, conforme o exemplo abaixo (dê o nome que preferir). Utilize camadas de escalonamento e *reshape* conforme necessário.

#### Dicas
- Funções úteis: `tf.keras.layers.experimental.preprocessing.Rescaling`, `tf.keras.layers.Reshape`
- Camadas convolucionais 2D exigem que a entrada seja um tensor 3D, sendo o último eixo correspondente ao número de canais (no caso, apenas 1, para uma imagem em tons de cinza).

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def make_model():
    model = keras.Sequential([
        keras.Input(shape=(28, 28)),
        layers.Rescaling(1.0 / 255.0),
        layers.Reshape((28, 28, 1)),

        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(128, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])

    return model

make_model().summary()

3. Desenvolva (i.e., aprimore a arquitetura) e treine sua rede (a partir do zero), tentando conseguir uma acurácia de validação de pelo menos 99.2%. (Lembre que usando apenas camadas densas é difícil conseguir uma acurácia muito superior a 98%.) Em seguida, calcule a acurácia no conjunto de teste.


In [ ]:
model = make_model()

# A saída do modelo usa softmax, portanto NÃO são logits.
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(x_val, y_val)
)

plots(history)

4. (OPCIONAL) Por que o uso de Dropout faz com que o desempenho de treinamento comece bastante inferior ao de validação?

#### Dicas
- Parta da arquitetura deste [tutorial](https://keras.io/examples/vision/mnist_convnet) (com os devidos ajustes feitos no item anterior) e adicione uma camada densa com um número suficiente de unidades. Lembre-se de (ao contrário do tutorial) trazer para dentro do modelo qualquer pré-processamento necessário.
- Visualize os gráficos do treinamento usando a função `plots` fornecida (ou a ferramenta TensorBoard).
- Ao usar camadas convolucionais com GPU, a execução paralelizada torna impossível garantir a reproducibilidade, portanto, não perca tempo com isso.

## (OPCIONAL) Fashion-MNIST

5. (OPCIONAL) Repita para o conjunto [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist), o qual também está [disponível no Keras](https://keras.io/api/datasets/fashion_mnist). Nesse caso, é suficiente aproveitar a mesma arquitetura do modelo e apenas (se necessário) alterar a taxa de aprendizado e o número de épocas. Sem muito esforço é possível conseguir uma acurácia de validação de 92% (em comparação com 87% para uma rede densa). Se desejar, visualize algumas imagens do conjunto de treinamento e algumas predições erradas no conjunto de teste.

### Observação sobre esta versão

Os datasets CIFAR-10 e Cats vs Dogs são carregados a partir de arquivos previamente baixados e armazenados no Google Drive. Assim, o notebook não depende do download automático dos datasets durante a execução.

**Estrutura sugerida no Google Drive:**
```text
Meu Drive/
└── datasets/
    ├── cifar-10-python.tar.gz
    └── kagglecatsanddogs_3367a.zip
```


## (OPCIONAL) Fashion-MNIST

O Fashion-MNIST possui imagens em tons de cinza com o mesmo tamanho do MNIST
(28 × 28 pixels) e 10 classes. A mesma ideia de arquitetura convolucional pode
ser reutilizada, alterando apenas os dados e, se necessário, os hiperparâmetros.

Neste item, vamos:
1. carregar o Fashion-MNIST;
2. separar 5.000 imagens para validação;
3. reutilizar uma CNN semelhante à usada no MNIST;
4. treinar a rede;
5. avaliar no conjunto de teste;
6. visualizar algumas classificações incorretas.

In [ ]:
from tensorflow.keras.datasets import fashion_mnist

(x_fashion_train, y_fashion_train), (x_fashion_test, y_fashion_test) = (
    fashion_mnist.load_data()
)

print("Train:", x_fashion_train.shape, y_fashion_train.shape)
print("Test :", x_fashion_test.shape, y_fashion_test.shape)

In [ ]:
# Últimas 5.000 imagens para validação.
x_fashion_val = x_fashion_train[55000:]
y_fashion_val = y_fashion_train[55000:]

x_fashion_train = x_fashion_train[:55000]
y_fashion_train = y_fashion_train[:55000]

print("Train:", x_fashion_train.shape, y_fashion_train.shape)
print("Val  :", x_fashion_val.shape, y_fashion_val.shape)
print("Test :", x_fashion_test.shape, y_fashion_test.shape)

In [ ]:
fashion_model = make_model()

fashion_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

fashion_model.summary()

In [ ]:
fashion_history = fashion_model.fit(
    x_fashion_train,
    y_fashion_train,
    epochs=20,
    batch_size=32,
    validation_data=(x_fashion_val, y_fashion_val)
)

plots(fashion_history)

In [ ]:
fashion_test_loss, fashion_test_acc = fashion_model.evaluate(
    x_fashion_test,
    y_fashion_test,
    verbose=1
)

print(f"Loss no teste: {fashion_test_loss:.4f}")
print(f"Acurácia no teste: {fashion_test_acc:.4f}")

In [ ]:
# Visualização de algumas classificações incorretas.
fashion_pred = fashion_model.predict(x_fashion_test, verbose=0)
fashion_pred_labels = np.argmax(fashion_pred, axis=1)

wrong = np.where(fashion_pred_labels != y_fashion_test)[0]

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

plt.figure(figsize=(10, 10))

for i, idx in enumerate(wrong[:9]):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(x_fashion_test[idx], cmap="gray")
    plt.title(
        f"Real: {class_names[y_fashion_test[idx]]}\n"
        f"Pred: {class_names[fashion_pred_labels[idx]]}"
    )
    plt.axis("off")

plt.tight_layout()

## CIFAR-10 — carregamento externo pelo Google Drive

Nesta versão, o dataset não é baixado pelo Keras. O arquivo oficial é baixado externamente, enviado ao Google Drive e carregado pelo Colab.


In [ ]:
# ============================================================
# CIFAR-10 — arquivo externo carregado pelo Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DATASET_DIR = "/content/drive/MyDrive/datasets"
CIFAR_ARCHIVE = os.path.join(
    DRIVE_DATASET_DIR,
    "cifar-10-python.tar.gz"
)

CIFAR_EXTRACT_DIR = "/content/cifar-10-batches-py"

if not os.path.exists(CIFAR_ARCHIVE):
    raise FileNotFoundError(
        f"\nArquivo não encontrado:\n{CIFAR_ARCHIVE}\n\n"
        "Baixe externamente o arquivo cifar-10-python.tar.gz "
        "e coloque-o na pasta datasets do Google Drive."
    )

if not os.path.exists(CIFAR_EXTRACT_DIR):
    with tarfile.open(CIFAR_ARCHIVE, "r:gz") as tar:
        tar.extractall("/content")

def load_cifar_batch(path):
    with open(path, "rb") as f:
        batch = pickle.load(f, encoding="bytes")

    x = batch[b"data"].reshape(
        -1, 3, 32, 32
    ).transpose(0, 2, 3, 1)

    y = np.asarray(batch[b"labels"], dtype=np.int64)

    return x, y

x_parts = []
y_parts = []

for i in range(1, 6):
    x, y = load_cifar_batch(
        os.path.join(CIFAR_EXTRACT_DIR, f"data_batch_{i}")
    )
    x_parts.append(x)
    y_parts.append(y)

x_train = np.concatenate(x_parts, axis=0)
y_train = np.concatenate(y_parts, axis=0)

x_test, y_test = load_cifar_batch(
    os.path.join(CIFAR_EXTRACT_DIR, "test_batch")
)

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)
print("y_test :", y_test.shape, y_test.dtype)

Observe que o `shape` do array `y` precisa ser corrigido:
- Para usar a perda `sparse_categorical_crossentropy`, `y` precisa ser um tensor 1D com valores em `[0, 1, ..., n_classes-1]`
- Para usar a perda `categorical_crossentropy`, `y` precisa ser um tensor 2D com *shape* `(n_samples, n_classes)` e codificação *one-hot*


In [ ]:
# Os rótulos do CIFAR-10 são inteiros de 0 a 9.
# Esse formato é adequado para SparseCategoricalCrossentropy.
y_train = y_train.reshape(-1)
y_test = y_test.reshape(-1)

# Últimas 5.000 imagens do treinamento para validação.
x_val = x_train[45000:]
y_val = y_train[45000:]

x_train = x_train[:45000]
y_train = y_train[:45000]

print("Train:", x_train.shape, y_train.shape)
print("Val  :", x_val.shape, y_val.shape)
print("Test :", x_test.shape, y_test.shape)

In [ ]:
plt.figure(figsize=(12,6))
for i in range(5):
  for c in range(10):
    plt.subplot(5, 10, 10*i+c+1)
    img = x_train[y_train == c][i]
    plt.imshow(img)
    if i == 0:
      plt.title('y = {}'.format(c))
    plt.axis('off')

In [ ]:
num_classes = 10

def make_model_CIFAR():
    model = keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Rescaling(1.0 / 255.0),

        layers.Conv2D(32, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),

        layers.Conv2D(64, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),

        layers.Conv2D(128, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),

        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ])

    return model

make_model_CIFAR().summary()

6. Inicialmente, apenas converta a mesma arquitetura utilizada no MNIST para o formato das imagens do CIFAR-10 e treine o modelo. Note que agora não é mais necessário usar uma camada `Reshape`. Certifique-se de escolher um batch size e taxa de aprendizado apropriadas. Observe que é difícil obter uma acurácia de validação superior a 73%.

7. Por que você acha que isso acontece? Explique.

In [ ]:
model = make_model_CIFAR()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(x_train, y_train, epochs=20, batch_size=64, validation_data=(x_val, y_val))

### Resposta
7. A acurácia de validação fica entorno de 75%, mesmo com 20 épocas, devido a ao rápido overffiting que o modelo está gerando, provavelmente pela rede não estar bem estruturada ou não profunda o suficiente para lidar com imagens pequenas e com muita infomarção espacial.

### Data augmentation
Para melhorar o desempenho, utilizaremos a técnica de aumento de dados (*data augmentation*). Há duas formas principais de utilizar esta técnica no Keras:
- Usando a função [`tf.keras.preprocessing.image.ImageDataGenerator`](https://keras.io/api/preprocessing/image/#imagedatagenerator-class), a qual opcionalmente permite aplicar transformações aleatórias. Esta é a abordagem mais tradicional.
- Usando camadas de *data augmentation* como parte do modelo, as quais aplicam transformações aleatórias *somente* durante o treinamento (ficando inativas fora do treinamento). Esta é uma abordagem mais recente e ainda experimental. A principal vantagem é tornar as operações mais rápidas pelo uso da GPU, conforme explicado [neste tutorial](https://keras.io/guides/preprocessing_layers/#preprocessing-data-before-the-model-or-inside-the-model) e também [neste](https://keras.io/examples/vision/image_classification_from_scratch/#two-options-to-preprocess-the-data).

Para acelerar o treinamento usaremos aqui a segunda abordagem. Um exemplo é mostrado abaixo; você pode adicionar outras transformações se desejar.



In [ ]:
from tensorflow.keras import Sequential


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomTranslation(
        height_factor=0.1,
        width_factor=0.1
    ),
    layers.RandomFlip("horizontal"),
], name="data_augmentation")

print(data_augmentation)

In [ ]:
def make_model_CIFAR_augmented():
    model = keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        data_augmentation,
        layers.Rescaling(1.0 / 255.0),

        layers.Conv2D(32, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),

        layers.Conv2D(64, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),

        layers.Conv2D(128, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),

        layers.Dropout(0.5),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ])

    return model

make_model_CIFAR_augmented().summary()

**Obs:** Aparentemente há um bug ainda não resolvido nas camadas `RandomTranslation` e `RandomRotation`, conforme descrito [aqui](https://stackoverflow.com/questions/62339559/black-pixels-outside-the-border-when-using-keras-layers-experimental-preprocessi) e visualizado por exemplo [aqui](https://keras.io/examples/vision/image_classification_from_scratch/#using-image-data-augmentation), [aqui](https://keras.io/guides/transfer_learning/#using-random-data-augmentation) e nas imagens abaixo. (Não deveria haver bordas pretas nas imagens transformadas.) Felizmente esse defeito não afeta o desempenho de forma significativa.

In [ ]:
plt.figure(figsize=(12, 8))

for j in range(15):
    i = j

    img = data_augmentation(
        x_train[i:i+1],
        training=True
    )[0].numpy()

    plt.subplot(3, 5, j + 1)
    plt.imshow(np.clip(img, 0, 255).astype("uint8"))
    plt.title(f"Classe: {y_train[i]}")
    plt.axis("off")

plt.tight_layout()

8. Treine o modelo até obter pelo menos 80% de acurácia de validação. Note que será preciso um número elevado de épocas (100 ou mais), o que deve levar vários minutos.

#### Dicas
- Certifique-se de usar uma taxa de aprendizado apropriada ao longo de todo o treinamento. Por exemplo:
 - Treine por N épocas com taxa constante, observe os resultados, depois continue o treinamento por mais N épocas, etc. Se em algum ponto o desempenho não estiver melhorando, reduza manualmente a taxa de aprendizado;
 - Utilize uma *callback* de decaimento, como [`ReduceLROnPlateau`](https://keras.io/api/callbacks/reduce_lr_on_plateau/) ou a genérica [`LearningRateScheduler`](https://keras.io/api/callbacks/learning_rate_scheduler/); ou
 - Utilize um valor pequeno constante e tenha bastante paciência.

-  Fique à vontade para aprimorar o modelo se desejar. Para facilitar, você pode se basear em quaisquer outras referências ou tutoriais disponíveis, como por exemplo: [Tutorial 1](https://www.learnopencv.com/image-classification-using-convolutional-neural-networks-in-keras) ou [Tutorial 2](https://machinelearningmastery.com/how-to-develop-a-cnn-from-scratch-for-cifar-10-photo-classification). Note que alguns tutoriais estão desatualizados; por exemplo, a função `fit_generator` tornou-se obsoleta, tendo sido incorporada à função `fit`. No entanto, esteja ciente de que não é necessário um modelo muito complexo para obter a acurácia desejada.

In [ ]:
model = make_model_CIFAR_augmented()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.002),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(x_train, y_train, epochs=25, batch_size=32, validation_data=(x_val, y_val))

# 2. Usando uma rede pré-treinada (*transfer learning & fine-tuning*)

Nesta versão, o conjunto Cats vs Dogs também é baixado externamente e carregado a partir de um arquivo ZIP armazenado no Google Drive.


### Primeiro Dataset

In [ ]:
# ============================================================
# Cats vs Dogs — arquivo externo carregado pelo Google Drive
# ============================================================

# O Google Drive já foi montado na seção anterior.
# Caso execute esta seção isoladamente, descomente:
#
# from google.colab import drive
# drive.mount("/content/drive")

DRIVE_DATASET_DIR = "/content/drive/MyDrive/datasets"

CATS_DOGS_ZIP = os.path.join(
    DRIVE_DATASET_DIR,
    "kagglecatsanddogs_3367a.zip"
)

CATS_DOGS_ROOT = "/content/PetImages"

if not os.path.exists(CATS_DOGS_ZIP):
    raise FileNotFoundError(
        f"\nArquivo não encontrado:\n{CATS_DOGS_ZIP}\n\n"
        "Baixe externamente o ZIP do Cats vs Dogs "
        "e coloque-o na pasta datasets do Google Drive."
    )

if not os.path.exists(CATS_DOGS_ROOT):
    with zipfile.ZipFile(CATS_DOGS_ZIP, "r") as zip_ref:
        zip_ref.extractall("/content")

print("Dataset:", CATS_DOGS_ROOT)
print(
    "Cats:",
    len(os.listdir(os.path.join(CATS_DOGS_ROOT, "Cat")))
)
print(
    "Dogs:",
    len(os.listdir(os.path.join(CATS_DOGS_ROOT, "Dog")))
)

In [ ]:
print("Arquivos em PetImages:")
print("Cat:", len(os.listdir("/content/PetImages/Cat")))
print("Dog:", len(os.listdir("/content/PetImages/Dog")))

In [ ]:
# Remove arquivos que não podem ser identificados como imagens JPEG.
num_skipped = 0

for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join(
        CATS_DOGS_ROOT,
        folder_name
    )

    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)

        if not os.path.isfile(fpath):
            continue

        try:
            with open(fpath, "rb") as f:
                is_jfif = tf.compat.as_bytes("JFIF") in f.peek(10)
        except Exception:
            is_jfif = False

        if not is_jfif:
            num_skipped += 1
            os.remove(fpath)

print(f"Arquivos inválidos removidos: {num_skipped}")

In [ ]:
image_size = (180, 180)
batch_size = 32
seed = 1337

train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/PetImages",
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="binary",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/PetImages",
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="binary",
)

print("Classes:", train_ds.class_names)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(int(labels[i]))
        plt.axis("off")


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
], name="data_augmentation")

plt.figure(figsize=(10, 10))

for images, _ in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)

        augmented_image = data_augmentation(
            tf.expand_dims(images[i], 0),
            training=True
        )[0]

        plt.imshow(
            tf.cast(
                tf.clip_by_value(
                    augmented_image, 0, 255
                ),
                tf.uint8
            )
        )

        plt.axis("off")

plt.tight_layout()

In [ ]:
train_ds = train_ds.prefetch(buffer_size=32)
val_ds = val_ds.prefetch(buffer_size=32)

### Conjunto Cats vs Dogs — divisão treino/validação/teste

Nesta alternativa, o dataset é baixado externamente, armazenado no Google Drive
e carregado localmente no Colab. A divisão abaixo cria aproximadamente 70% para
treinamento, 15% para validação e 15% para teste, usando uma única semente.

In [ ]:
image_size = (150, 150)
batch_size = 32
seed = 1337

# 70% treinamento e 30% temporário.
train_ds = tf.keras.utils.image_dataset_from_directory(
    CATS_DOGS_ROOT,
    validation_split=0.30,
    subset="training",
    seed=seed,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=True,
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    CATS_DOGS_ROOT,
    validation_split=0.30,
    subset="validation",
    seed=seed,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=False,
)

# Aproximadamente 15% validação + 15% teste.
temp_batches = tf.data.experimental.cardinality(temp_ds).numpy()
val_batches = temp_batches // 2

validation_ds = temp_ds.take(val_batches)
test_ds = temp_ds.skip(val_batches)

print("Classes:", train_ds.class_names)
print("Batches de treino:", tf.data.experimental.cardinality(train_ds).numpy())
print("Batches de validação:", tf.data.experimental.cardinality(validation_ds).numpy())
print("Batches de teste:", tf.data.experimental.cardinality(test_ds).numpy())

In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)

        plt.imshow(
            images[i].numpy().astype("uint8")
        )

        label = int(labels[i].numpy())
        plt.title("Cat" if label == 0 else "Dog")
        plt.axis("off")

plt.tight_layout()

In [ ]:
# Prefetch melhora o fluxo de dados durante o treinamento.
# O cache em memória é evitado para não consumir muita RAM.
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
validation_ds = validation_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 10))
    first_image = images[0]

    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        augmented_image = data_augmentation(
            tf.expand_dims(first_image, 0),
            training=True
        )
        plt.imshow(tf.cast(augmented_image[0], tf.uint8))
        plt.title("Cat" if int(labels[0]) == 0 else "Dog")
        plt.axis("off")

    plt.tight_layout()

### Modelo

In [ ]:
from tensorflow.keras.applications.resnet_v2 import (
    ResNet50V2,
    preprocess_input
)

base_model = ResNet50V2(
    include_top=False,
    weights="imagenet",
    input_shape=(150, 150, 3),
    pooling="avg"
)

# Primeira etapa: somente a nova camada classificadora será treinada.
base_model.trainable = False

inputs = keras.Input(shape=(150, 150, 3))

x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)

x = layers.Dropout(0.2)(x)

# Uma saída linear + BinaryCrossentropy(from_logits=True).
outputs = layers.Dense(1)(x)

model = keras.Model(inputs, outputs)

model.summary()

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(0.002),
              loss=keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=[keras.metrics.BinaryAccuracy()])

epochs = 10
model.fit(train_ds, epochs=epochs, validation_data=validation_ds)

In [ ]:
history_transfer = model.fit(
    train_ds,
    epochs=10,
    validation_data=validation_ds
)

plots(history_transfer)

## Fine-Tuning

Depois do treinamento inicial, algumas das últimas camadas da ResNet50V2
são liberadas para um ajuste fino com uma taxa de aprendizado pequena.

In [ ]:
# Libera a rede pré-treinada para fine-tuning.
base_model.trainable = True

# Mantém as primeiras camadas congeladas.
fine_tune_at = 100

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy()]
)

fine_tune_epochs = 5

history_fine = model.fit(
    train_ds,
    epochs=fine_tune_epochs,
    validation_data=validation_ds
)

plots(history_fine)

In [ ]:
# Avaliação final no conjunto de teste
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"Acurácia no teste: {test_acc:.4f}")


# 3. (OPCIONAL) Visualizando os padrões aprendidos

1. (OPCIONAL) Para alguma rede convolucional à sua escolha, visualize o que cada camada da rede "aprendeu"; mais precisamente, mostre exemplos de imagens de entrada que maximizam a ativação dos filtros em cada camada. Para isso, siga este [tutorial](https://keras.io/examples/vision/visualizing_what_convnets_learn/).

# 3. Visualização dos padrões aprendidos

Nesta etapa vamos investigar visualmente o que as camadas convolucionais
aprenderam durante o treinamento.

A visualização será feita em duas partes:
- filtros da primeira camada convolucional;
- mapas de ativação de uma imagem de entrada.

Os filtros da primeira camada permitem observar padrões locais aprendidos,
enquanto os mapas de ativação mostram onde esses filtros respondem na imagem.

## 3.1 Visualizando os filtros da primeira camada convolucional

A primeira camada convolucional da CNN possui filtros pequenos. Em imagens
em tons de cinza, cada filtro possui apenas um canal de entrada.

In [ ]:
# Seleciona a primeira camada Conv2D da CNN do MNIST.
conv_layers = [
    layer for layer in model.layers
    if isinstance(layer, layers.Conv2D)
]

first_conv = conv_layers[0]
filters, biases = first_conv.get_weights()

print("Camada:", first_conv.name)
print("Shape dos filtros:", filters.shape)
print("Número de filtros:", filters.shape[-1])

In [ ]:
# Visualiza até 16 filtros da primeira camada.
num_filters = min(16, filters.shape[-1])

plt.figure(figsize=(10, 10))

for i in range(num_filters):
    ax = plt.subplot(4, 4, i + 1)

    filter_img = filters[:, :, 0, i]

    # Normalização apenas para facilitar a visualização.
    vmin = filter_img.min()
    vmax = filter_img.max()

    if vmax > vmin:
        filter_img = (filter_img - vmin) / (vmax - vmin)

    plt.imshow(filter_img, cmap="gray")
    plt.title(f"Filtro {i}")
    plt.axis("off")

plt.tight_layout()

## 3.2 Mapas de ativação

Agora vamos observar a resposta de uma camada convolucional para uma imagem
específica do conjunto de teste.

In [ ]:
# Escolhe uma imagem do conjunto de teste.
image_index = 0
sample = x_test[image_index:image_index + 1]

plt.figure(figsize=(4, 4))
plt.imshow(x_test[image_index], cmap="gray")
plt.title(f"Imagem de teste — classe real: {y_test[image_index]}")
plt.axis("off")
plt.show()

In [ ]:
# Modelo intermediário para obter a saída da primeira Conv2D.
activation_model = keras.Model(
    inputs=model.input,
    outputs=first_conv.output
)

activations = activation_model.predict(sample, verbose=0)

print("Shape das ativações:", activations.shape)

In [ ]:
num_activations = min(16, activations.shape[-1])

plt.figure(figsize=(10, 10))

for i in range(num_activations):
    ax = plt.subplot(4, 4, i + 1)
    plt.imshow(activations[0, :, :, i], cmap="gray")
    plt.title(f"Mapa {i}")
    plt.axis("off")

plt.tight_layout()

## 3.3 Interpretação

Os filtros representam parâmetros aprendidos pela primeira camada
convolucional. Os mapas de ativação mostram a resposta desses filtros
para uma determinada imagem.

**Questão:** observe os filtros e os mapas de ativação. Que tipos de
padrões locais parecem provocar respostas mais fortes? Relacione sua
observação com a função das camadas convolucionais no reconhecimento
de imagens.

### Questão opcional — Fashion-MNIST

Compare os resultados obtidos no MNIST e no Fashion-MNIST.

1. Compare as acurácias de validação e teste.
2. Observe as classificações incorretas.
3. Explique por que a mesma arquitetura pode apresentar desempenho diferente
   nos dois conjuntos.
4. Experimente alterar apenas a taxa de aprendizado ou o número de épocas e
   observe o efeito no treinamento.